In [ ]:
import pennylane as qp

In [ ]:
dev = qp.device("lightning.qubit", wires=3)


@qp.qjit(capture=True)
@qp.set_shots(10)
@qp.qnode(dev, mcm_method="one-shot")
def ghz():
    qp.Hadamard(0)
    qp.CNOT([0, 1])
    qp.CNOT([1, 2])
    return qp.sample([qp.measure(0), qp.measure(1), qp.measure(2)])


print("samples:", ghz())

In [ ]:
from placement import STEANE_CPU_DECODER_LIB_PATH

steane_decode = qp.CoprocessorFunction("steane_coprocessor", lib_path=STEANE_CPU_DECODER_LIB_PATH)

ctrl = qp.Controller(name="cpu-controller", device=qp.device("null.qubit", wires=3))
coproc = qp.Coprocessor(name="cpu-coproc", coprocessor_fn=steane_decode)
dev = qp.Backline(controller=ctrl, coprocessors=[coproc], transport="memcpy", qec_code="steane")


@qp.qjit(capture=True)
@qp.set_shots(10)
@qp.qnode(dev, mcm_method="one-shot")
def ghz():
    qp.Hadamard(0)
    qp.CNOT([0, 1])
    qp.CNOT([1, 2])
    return qp.sample([qp.measure(0), qp.measure(1), qp.measure(2)])


print("samples:", ghz())

In [ ]:
from placement import REMOTE_GPU_COPROC, REMOTE_VPK_CTRL

ctrl = qp.Controller(name="fpga-controller", **REMOTE_VPK_CTRL)
coproc = qp.Coprocessor(name="gpu-coproc", coprocessor_fn="gpu_steane_launcher", **REMOTE_GPU_COPROC)
dev = qp.Backline(controller=ctrl, coprocessors=[coproc], transport="rdma", qec_code="steane")


@qp.qjit(capture=True)
@qp.set_shots(10)
@qp.qnode(dev, mcm_method="one-shot")
def ghz():
    qp.Hadamard(0)
    qp.CNOT([0, 1])
    qp.CNOT([1, 2])
    return qp.sample([qp.measure(0), qp.measure(1), qp.measure(2)])


print("samples:", ghz())